# Lecture 1: exercise solutions

Worked solutions to the exercises in Lecture 1. Your own work may differ in the details (file, branch
and function names); what matters is the sequence of commands and why each one is needed.

## Exercise: a notebook in your repository, and `.gitignore`

After creating and running a notebook (say `analysis.ipynb`) in your repository, `git status --short` shows

```
?? .ipynb_checkpoints/
?? analysis.ipynb
```

Jupyter creates `.ipynb_checkpoints/` automatically: it holds autosaved copies of your notebooks and should
not be tracked. Create a file called `.gitignore` at the top level of the repository containing

```
.ipynb_checkpoints/
```

Now `git status --short` shows only the notebook and the `.gitignore` itself:

```
?? .gitignore
?? analysis.ipynb
```

Add and commit both, then push. Commit the `.gitignore` too, so the rule applies to everyone who clones the
repository:

```bash
git add analysis.ipynb .gitignore
git commit -a -m "Add analysis notebook and ignore notebook checkpoints"
git push
```

## Exercise: collaborating as a team

### Round 1: branches and pull requests

The owner of the chosen repository invites the others (*Settings*, then *Collaborators*, then *Add people*).
Then each member runs

```bash
git clone https://github.com/<owner>/<repository>.git
cd <repository>
git switch -c <your-name>-file
# create a file, e.g. notes_<your-name>.txt
git add notes_<your-name>.txt
git commit -a -m "Add notes_<your-name>.txt"
git push -u origin <your-name>-file
```

and opens a pull request for their branch on GitHub. Each branch adds a different file, so the pull requests
merge without conflicts. Once they are merged, everyone updates their local `main` and deletes the merged
branch: with the *Delete branch* button on the merged pull request (on GitHub), and locally with

```bash
git switch main
git pull
git branch -d <your-name>-file
```

### Round 2: a merge conflict

Each member creates a new branch from the up-to-date `main` and adds a different function to the end of the
**same** file (say `shared.py`):

```bash
git switch -c <your-name>-function
# add a function at the end of shared.py
git commit -a -m "Add <function name>"
git push -u origin <your-name>-function
```

Each opens a pull request. The first one merges without trouble. The others now report a conflict: every
branch added lines at the same place (the end of `shared.py`), and `main` has changed since they were
created, so git cannot tell what order the additions should go in.

To resolve it with `rebase`, the owner of each remaining branch runs, one branch at a time:

```bash
git switch main
git pull                        # fetch the work that has been merged
git switch <your-name>-function
git rebase main                 # replay your commit on top of the new main
```

Git stops with `CONFLICT (content): Merge conflict in shared.py`. In `shared.py` the conflicting region looks
like this (`HEAD` is `main`'s version; the other side is your commit being replayed):

```
<<<<<<< HEAD
def g():
    return 2
=======
def h():
    return 3
>>>>>>> 1a2b3c4 (Add h)
```

Keep both functions, with a blank line or two between them, delete the three marker lines, and save. Then

```bash
git add shared.py
git rebase --continue           # if an editor opens with the commit message, save and close it
git push --force-with-lease     # the rebase rewrote your branch, so a plain push is refused
```

The pull request updates itself and can now be merged; its `main` contains every function. Repeat for any
further branches, then update `main` locally and delete the merged branches as in round 1. (For a small
conflict like this, GitHub's *Resolve conflicts* button on the pull request is an alternative to rebasing.)

## Exercise: make a venv

```bash
python -m venv myenv            # or: python3 -m venv myenv
source myenv/bin/activate       # the prompt now starts with (myenv)
pip list
pip install matplotlib
pip list
deactivate
```

In a new environment `pip list` shows little more than pip itself:

```
Package Version
------- -------
pip     24.0
```

After `pip install matplotlib` it also lists matplotlib and the packages it depends on (`contourpy`, `cycler`,
`fonttools`, `kiwisolver`, `numpy`, `packaging`, `pillow`, `pyparsing`, ...), installed in `myenv/` only.
After `deactivate`, `python` is no longer the environment's.

## The conda exercise and "Finalizing our repository"

The complete solution is the `2026final` branch of
[`rhodrin/environments_mpm`](https://github.com/rhodrin/environments_mpm/tree/2026final). You can browse it
on GitHub, or compare it with your own work from inside your clone:

```bash
git fetch https://github.com/rhodrin/environments_mpm.git 2026final
git diff FETCH_HEAD
```

`envtest/builtins.py` contains the lecture's two functions plus one of our own that needs a new
dependency (the notes suggested a pandas example). Every function is listed in `__all__` so that the
package re-exports it:

```python
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter


__all__ = ['rand_array', 'smooth_image', 'my_mat_solve', 'array_summary']


def rand_array(shape):
    return np.random.rand(*shape)


def smooth_image(a, sigma=1):
    return gaussian_filter(a, sigma=sigma)


def my_mat_solve(A, b):
    return A.inv()*b


def array_summary(a):
    """Summary statistics (count, mean, std, min, quartiles, max) of each column of a 2D array."""
    return pd.DataFrame(a).describe()
```

The new script, `scripts/summarise_array.py`:

```python
from envtest import array_summary, rand_array

print(array_summary(rand_array((100, 3))))
```

`requirements.txt` lists every dependency:

```
numpy>=2.1
matplotlib>=3.10
scipy
scikit-image
sympy
pandas
```

so does `environment.yml`:

```yaml
name: mpm-lecture
channels:
  - conda-forge
  - nodefaults
dependencies:
  - python=3.14
  - pip
  - numpy>=2.1
  - matplotlib>=3.10
  - scipy
  - scikit-image
  - sympy
  - pandas
  - pip:
      - -e . # install local repo
```

and a `.gitignore` keeps the generated metadata out of the repository:

```
__pycache__/
*.egg-info/
.ipynb_checkpoints/
```

After committing and pushing, rebuilding the environment from scratch checks that the files really do
describe everything the code needs:

```bash
conda deactivate
conda remove --name mpm-lecture --all
conda env create -f environment.yml
conda activate mpm-lecture
python scripts/random_numpy_array.py
python scripts/smooth_image.py
python scripts/solve_matrix_equation.py
python scripts/summarise_array.py
```

All four scripts run in the rebuilt environment, and `git status` stays clean. Note that the dependency list
now lives in both `requirements.txt` and `environment.yml` (and partly in `pyproject.toml`): the last section
of the lecture explains why, for your own projects, `pyproject.toml` should be the single source of truth.